## Training demo for any model with custom loss function and SeismicDataLoader

In this demo, SeismicConvLSTM is used as the model and CustomLoss is used as the loss function.
- This demo uses train and validation dataloaders from dataloader.py. If you haven't seen dataloader_demo.ipynb, check it first.
- First, import train and validation dataloader
- Next, import custom loss function from customloss.py
- Next, we add SeismicConvLSTM model for demo purposes
- Next, import torch optimizer (we will use Adam, if you use other optimizers, import it)
- Then, we add matplotlib plt for traning, validation graph plotting
- Finally, add train_model function from trainer.py

In [2]:
import sys
import os
# Add the src_dataset directory to the sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../src_dataset/')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../utils/')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../src_model/')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../src_loss/')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from dataloader import train_dataloader, val_dataloader
from customLoss import CustomLoss
from convlstm import SeismicConvLSTM
import torch.optim as optim
import matplotlib.pyplot as plt
from trainer import train_model, train_model_with_logging

[I] Checking if the dataset is downloaded with the name afad_hfd5.zip
[W] Dataset is not found, downloading the dataset


KeyboardInterrupt: 

### Model setup

This below cell creates a model instance. If you'd like to use your own model, change this cell.
The example SeismicConvLSTM model takes the below inputs:
- input channel size
- output dimension
- numer of sequence windows
- data points in each window
- hidden dimensions of convlstm cells
- kernel size of conv layers
- number of convlstm cells

model variable is used to keep the model

In [ ]:
# Define the model
# We are dynamically fetching the input channels, num_windows, data_points_of_each_window, and the output_dim from the dataset using the dataloader
# Create an iterator from the dataloader
train_iter = iter(train_dataloader)
# Get the first batch
one_sample = next(train_iter) # one_sample[0].shape, one_sample[1].shape --> torch.Size([16, 3, 3, 1, 900]) torch.Size([16, 6])

INPUT_CHANNELS = one_sample[0].shape[2] # one_sample[0].shape[2] --> 3
OUTPUT_DIM = one_sample[1].shape[1] # one_sample[1].shape[1] --> 6
NUM_WINDOWS = one_sample[0].shape[1] # one_sample[0].shape[1] --> 3
DATA_POINTS_OF_EACH_WINDOW = one_sample[0].shape[4] # one_sample[0].shape[4] --> 900
HIDDEN_DIM = [32, 32, 64, 64, 128]
KERNEL_SIZE = (11, 5, 3, 3, 3)   
NUM_CONVLSTM_BLOCKS = 5
print("output_dim: ", OUTPUT_DIM)
assert len(HIDDEN_DIM) == NUM_CONVLSTM_BLOCKS, "The number of hidden dimensions should be equal to the number of ConvLSTM blocks"
model = SeismicConvLSTM(input_dim=INPUT_CHANNELS, hidden_dim=HIDDEN_DIM, kernel_size=KERNEL_SIZE, num_layers=NUM_CONVLSTM_BLOCKS, output_dim=OUTPUT_DIM, num_windows = NUM_WINDOWS, data_points = DATA_POINTS_OF_EACH_WINDOW)

print(model)

### Training the model
We have been created dataloaders and model. Now we will train the model with generic train function.
- First, initialize optimizer. In this demo, Adam optimizer is used with lr = LEARNING_RATE.
- Next, set the loss function as criterion. In this demo, CustomLoss function with lambda_val = CUSTOM_LOSS_LAMBDA is used.
- Then, set the number of epochs.
- Next, train the model. train_model function trains the model with train_dataloader, and validates after each epoch with val_dataloader

In [ ]:
LEARNING_RATE = 0.1
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

CUSTOM_LOSS_LAMBDA = 2
criterion = CustomLoss(lambda_val=CUSTOM_LOSS_LAMBDA)

num_epochs = 30


# Train the model with the training dataloader
model, train_losses, val_losses = train_model_with_logging(model, train_dataloader, val_dataloader, criterion, optimizer, num_epochs, 'train_log.csv') 


In [ ]:
import os
import time
import torch

os.makedirs('models', exist_ok=True)
date_time_str = time.strftime("%Y%m%d-%H%M")
model_name = 'models/model_{}_epoch{}_lr{}_lambda{}_windows{}_datapoints{}.pth'.format(date_time_str, num_epochs, LEARNING_RATE, CUSTOM_LOSS_LAMBDA, NUM_WINDOWS, DATA_POINTS_OF_EACH_WINDOW)
torch.save(model.state_dict(), model_name)
print('Model saved at: ', model_name)

# plot and save the training and validation losses
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig('models/Train_and_val_loss_model_{}'.format(model_name.split('/')[1].split('.')[0]))
plt.show()